In [1]:
!pip install deap
import random
import operator
from deap import gp, creator, base, tools, algorithms
import numpy as np

In [2]:
# Funciones lógicas (con protección contra errores)
def and_(a, b): return int(bool(a) and bool(b))
def or_(a, b):  return int(bool(a) or bool(b))
def not_(a):    return int(not bool(a))
def xor_(a, b): return int(bool(a) ^ bool(b))
def nand_(a, b): return int(not (bool(a) and bool(b)))
def nor_(a, b):  return int(not (bool(a) or bool(b)))

# Crear el conjunto de primitivas
pset = gp.PrimitiveSet("MAIN", 4)  # 4 entradas: A, B, C, D
pset.addPrimitive(and_,  2, name="AND")
pset.addPrimitive(or_,   2, name="OR")
pset.addPrimitive(not_,  1, name="NOT")
pset.addPrimitive(xor_,  2, name="XOR")
pset.addPrimitive(nand_, 2, name="NAND")
pset.addPrimitive(nor_,  2, name="NOR")
pset.renameArguments(ARG0='A', ARG1='B', ARG2='C', ARG3='D')

# ─── 2. TABLA DE VERDAD DEL SEGMENTO 'a' ──────────────────────────────────────
# Segmento 'a' se enciende para: 0,2,3,5,6,7,8,9

TABLA_SEGMENTO_A = [
    # (A, B, C, D, seg_a)
    (0, 0, 0, 0, 1),  # 0 → ON
    (0, 0, 0, 1, 0),  # 1 → OFF
    (0, 0, 1, 0, 1),  # 2 → ON
    (0, 0, 1, 1, 1),  # 3 → ON
    (0, 1, 0, 0, 0),  # 4 → OFF
    (0, 1, 0, 1, 1),  # 5 → ON
    (0, 1, 1, 0, 1),  # 6 → ON
    (0, 1, 1, 1, 1),  # 7 → ON
    (1, 0, 0, 0, 1),  # 8 → ON
    (1, 0, 0, 1, 1),  # 9 → ON
]

# ─── 3. FUNCIÓN DE APTITUD ─────────────────────────────────────────────────────

def evaluar_circuito(individuo):
    """Cuenta cuántos casos el árbol clasifica correctamente."""
    funcion = gp.compile(individuo, pset)
    correctos = sum(
        1 for A, B, C, D, esperado in TABLA_SEGMENTO_A
        if funcion(A, B, C, D) == esperado
    )
    return (correctos,)

# ─── 4. CONFIGURAR PG CON DEAP ─────────────────────────────────────────────────

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("expr",       gp.genHalfAndHalf, pset=pset, min_=1, max_=4)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluar_circuito)
toolbox.register("select",   tools.selTournament, tournsize=3)
toolbox.register("mate",     gp.cxOnePoint)
toolbox.register("expr_mut", gp.genFull, min_=0, max_=2)
toolbox.register("mutate",   gp.mutUniform, expr=toolbox.expr_mut, pset=pset)

# Limitar profundidad máxima del árbol (evitar bloat)
toolbox.decorate("mate",   gp.staticLimit(key=operator.attrgetter("height"), max_value=17))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=17))

# ─── 5. EJECUTAR PG ────────────────────────────────────────────────────────────

def ejecutar_pg_circuito():
    random.seed(42)
    pop = toolbox.population(n=200)
    hof = tools.HallOfFame(1)  # Mejor individuo

    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("max",  np.max)
    stats.register("avg",  np.mean)

    pop, log = algorithms.eaSimple(
        pop, toolbox,
        cxpb=0.7,    # probabilidad de cruce
        mutpb=0.01,  # probabilidad de mutación
        ngen=300,    # generaciones
        stats=stats,
        halloffame=hof,
        verbose=True
    )

    mejor = hof[0]
    print(f"\n✅ Mejor solución encontrada:")
    print(f"   Expresión: {mejor}")
    print(f"   Aptitud:   {mejor.fitness.values[0]}/10 casos correctos")

    # Verificar
    funcion = gp.compile(mejor, pset)
    print("\n📋 Verificación:")
    for A, B, C, D, esperado in TABLA_SEGMENTO_A:
        resultado = funcion(A, B, C, D)
        estado = "✅" if resultado == esperado else "❌"
        print(f"   N={8*A+4*B+2*C+D}: esperado={esperado}, obtenido={resultado} {estado}")

    return mejor

if __name__ == "__main__":
    ejecutar_pg_circuito()

gen	nevals	max	avg  
0  	200   	9  	5.195
1  	144   	9  	6.335
2  	154   	9  	7.08 
3  	158   	9  	7.53 
4  	146   	9  	7.68 
5  	145   	9  	7.68 
6  	131   	9  	7.82 
7  	147   	9  	8.04 
8  	148   	10 	8.06 
9  	147   	10 	8.505
10 	120   	10 	8.525
11 	135   	10 	8.595
12 	138   	10 	8.79 
13 	128   	10 	9.03 
14 	137   	10 	9.32 
15 	129   	10 	9.325
16 	133   	10 	9.4  
17 	141   	10 	9.325
18 	137   	10 	9.38 
19 	134   	10 	9.385
20 	143   	10 	9.475
21 	145   	10 	9.43 
22 	137   	10 	9.455
23 	130   	10 	9.465
24 	141   	10 	9.57 
25 	135   	10 	9.55 
26 	163   	10 	9.48 
27 	137   	10 	9.615
28 	134   	10 	9.55 
29 	166   	10 	9.645
30 	142   	10 	9.67 
31 	131   	10 	9.63 
32 	151   	10 	9.7  
33 	136   	10 	9.705
34 	145   	10 	9.74 
35 	132   	10 	9.675
36 	131   	10 	9.72 
37 	151   	10 	9.675
38 	136   	10 	9.76 
39 	135   	10 	9.7  
40 	134   	10 	9.755
41 	150   	10 	9.715
42 	146   	10 	9.7  
43 	133   	10 	9.74 
44 	136   	10 	9.765
45 	142   	10 	9.66 
46 	131   	10